In [1]:
import os
import yfinance as yf 
import pandas as pd

In [2]:
universe = pd.read_csv("../data/raw/universe.csv")
universe["Ticker"].duplicated().sum()
print(universe)

                       Company  Ticker Country   Band  \
0                     RELX plc   REL.L      UK   Mega   
1                 Experian plc  EXPN.L      UK  Large   
2        Auto Trader Group plc  AUTO.L      UK    Mid   
3                Rightmove plc   RMV.L      UK    Mid   
4                       RM plc    RM.L      UK  Small   
..                         ...     ...     ...    ...   
122                Cintas Corp    CTAS      US  Large   
123    Jack Henry & Associates    JKHY      US    Mid   
124                Rollins Inc     ROL      US  Large   
125              UniFirst Corp     UNF      US    Mid   
126  Healthcare Services Group    HCSG      US  Small   

                        Sector  \
0         Information Services   
1         Information Services   
2         Information Services   
3         Information Services   
4         Information Services   
..                         ...   
122  Mission-Critical Services   
123  Mission-Critical Services   
124  Miss

## Data Collection


In [3]:
START_DATE = "2021-01-01"
END_DATE = "2025-12-31"

In [ ]:
log_entries = []

for ticker in universe["Ticker"]:
    ticker_object = yf.Ticker(ticker)
    try:
        prices = ticker_object.history(start = START_DATE, end = END_DATE, auto_adjust = False)
        if prices.empty:
            log_entries.append({
                "ticker": ticker, 
                "data_type": "prices",
                "status": "fail",
                "rows": 0,
                "start_date": None,
                "end_date": None,
                "missing_values": None,
                "missing_required_fields": None,
                "error": "empty data returned"
            })
        else:
            prices.to_csv(f"../data/raw/prices/{ticker}.csv")
            missing_values = prices.isna().sum().sum()
            log_entries.append({
                "ticker": ticker, 
                "data_type": "prices",
                "status": "success",
                "rows": len(prices),
                "start_date": prices.index.min(),
                "end_date": prices.index.max(),
                "missing_values": missing_values,
                "missing_required_fields": None,
                "error": None
            })
    except Exception as e:
        log_entries.append({
            "ticker": ticker, 
            "data_type": "prices",
            "status": "fail",
            "rows": 0,
            "start_date": None,
            "end_date": None,
            "missing_values": None,
            "missing_required_fields": None,
            "error": str(e)
        })

collection_log = pd.DataFrame(log_entries)
collection_log.to_csv("../data/raw/collection_log.csv", index = False)

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: DNB"}}}
$DNB: possibly delisted; no timezone found
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: SXS.L"}}}
$SXS.L: possibly delisted; no timezone found
$TET.L: possibly delisted; no price data found  (1d 2021-01-01 -> 2025-12-31) (Yahoo error = "Data doesn't exist for startDate = 1609459200, endDate = 1767139200")
$RWI.L: possibly delisted; no timezone found
$MEG: possibly delisted; no timezone found
$APH.L: possibly delisted; no timezone found
$ALPH.L: possibly delisted; no timezone found
$FI: possibly delisted; no timezone found


In [ ]:
cash_flow_required_fields = [
    "Operating Cash Flow",
    "Capital Expenditure",
    "Depreciation And Amortization",
    "Free Cash Flow",
    "Stock Based Compensation"
]

log_entries = []

for ticker in universe["Ticker"]:
    try:
        ticker_object = yf.Ticker(ticker)
        cash_flow = ticker_object.cash_flow
        available_cash_flow_fields = [field for field in cash_flow_required_fields if field in cash_flow.index]
        filtered_cash_flow = cash_flow.loc[available_cash_flow_fields]
        if filtered_cash_flow.empty:
            log_entries.append({
                "ticker": ticker, 
                "data_type": "cash_flow",
                "status": "fail",
                "rows": 0,
                "start_date": None,
                "end_date": None,
                "missing_values": None,
                "missing_required_fields": None,
                "error": "empty data returned"
            })
        else:
            os.makedirs(f"../data/raw/fundamentals/{ticker}", exist_ok=True)
            cleaned_cash_flow = filtered_cash_flow.dropna(axis = 1, how = "all")
            cleaned_cash_flow.to_csv(f"../data/raw/fundamentals/{ticker}/cash_flow.csv")
            missing_values = cleaned_cash_flow.isna().sum().sum()
            missing_fields = [field for field in cash_flow_required_fields if field not in cleaned_cash_flow.index]
            log_entries.append({
                "ticker": ticker, 
                "data_type": "cash_flow",
                "status": "success",
                "rows": cleaned_cash_flow.shape[1],
                "start_date": cleaned_cash_flow.columns.min(),
                "end_date": cleaned_cash_flow.columns.max(),
                "missing_values": missing_values,
                "missing_required_fields": missing_fields,
                "error": None
            })
    except Exception as e:
        log_entries.append({
            "ticker": ticker, 
            "data_type": "cash_flow",
            "status": "fail",
            "rows": 0,
            "start_date": None,
            "end_date": None,
            "missing_values": None,
            "missing_required_fields": None,
            "error": str(e)
        })

collection_log = pd.DataFrame(log_entries)
collection_log.to_csv("../data/raw/collection_log.csv", index = False)

In [ ]:
metadata_entries = []
log_entries = []

for ticker in universe["Ticker"]:
    try:
        ticker_object = yf.Ticker(ticker)
        info = ticker_object.info
        currency = info.get("currency")
        shares_outstanding = info.get("sharesOutstanding")
        exchange = info.get("exchange")
        industry = info.get("industry")

        missing_fields = [field for field, value in {
            "currency": currency,
            "sharesOutstanding": shares_outstanding,
            "exchange": exchange,
            "industry": industry
        }.items() if value is None]

        num_missing_fields = len(missing_fields)
        if num_missing_fields == 4:
            status = "fail"
        elif num_missing_fields == 0:
            status = "success"
        else:
            status = "partial"

        log_entries.append({
            "ticker": ticker, 
            "data_type": "metadata",
            "status": status,
            "rows": None,
            "start_date": None,
            "end_date": None,
            "missing_values": None,
            "missing_required_fields": missing_fields,
            "error": None
        })

        if status != "fail":
            metadata_entries.append({
                "ticker": ticker,
                "currency": currency,
                "sharesOutstanding": shares_outstanding,
                "exchange": exchange,
                "industry": industry
            })
    except Exception as e:
        log_entries.append({
            "ticker": ticker, 
            "data_type": "metadata",
            "status": "fail",
            "rows": None,
            "start_date": None,
            "end_date": None,
            "missing_values": None,
            "missing_required_fields": None,
            "error": str(e)
        })

metadata_df = pd.DataFrame(metadata_entries)
metadata_df.to_csv("../data/raw/metadata.csv", index = False)

collection_log = pd.DataFrame(log_entries)
collection_log.to_csv("../data/raw/collection_log.csv", index = False)    

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: DNB"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: SXS.L"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: RWI.L"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: MEG"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: APH.L"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ALPH.L"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: FI"}}}


In [ ]:
balance_sheet_required_fields = [
    "Cash And Cash Equivalents",
    "Total Debt",
    "Current Debt",
    "Long Term Debt",
    "Total Assets",
    "Total Liabilities Net Minority Interest",
    "Total Equity Gross Minority Interest"
]

log_entries = []

for ticker in universe["Ticker"]:
    try:
        ticker_object = yf.Ticker(ticker)
        balance_sheet = ticker_object.balance_sheet
        available_balance_sheet_fields = [field for field in balance_sheet_required_fields if field in balance_sheet.index]
        filtered_balance_sheet = balance_sheet.loc[available_balance_sheet_fields]
        if filtered_balance_sheet.empty:
            log_entries.append({
                "ticker": ticker, 
                "data_type": "balance_sheet",
                "status": "fail",
                "rows": 0,
                "start_date": None,
                "end_date": None,
                "missing_values": None,
                "missing_required_fields": None,
                "error": "empty data returned"
            })
        else:
            os.makedirs(f"../data/raw/fundamentals/{ticker}", exist_ok = True)
            cleaned_balance_sheet = filtered_balance_sheet.dropna(axis = 1, how = "all")
            cleaned_balance_sheet.to_csv(f"../data/raw/fundamentals/{ticker}/balance_sheet.csv")
            missing_values = cleaned_balance_sheet.isna().sum().sum()
            missing_fields = [field for field in balance_sheet_required_fields if field not in cleaned_balance_sheet.index]
            log_entries.append({
                "ticker": ticker, 
                "data_type": "balance_sheet",
                "status": "success",
                "rows": cleaned_balance_sheet.shape[1],
                "start_date": cleaned_balance_sheet.columns.min(),
                "end_date": cleaned_balance_sheet.columns.max(),
                "missing_values": missing_values,
                "missing_required_fields": missing_fields,
                "error": None
            })
    except Exception as e:
        log_entries.append({
            "ticker": ticker, 
            "data_type": "balance_sheet",
            "status": "fail",
            "rows": 0,
            "start_date": None,
            "end_date": None,
            "missing_values": None,
            "missing_required_fields": None,
            "error": str(e)
        })

collection_log = pd.DataFrame(log_entries)
collection_log.to_csv("../data/raw/collection_log.csv", index = False)

In [4]:
income_statement_required_fields = [
    "Total Revenue",
    "Gross Profit",
    "Operating Income",
    "EBIT",
    "EBITDA",
    "Pretax Income",
    "Net Income",
    "Diluted EPS",
    "Interest Expense",
    "Tax Provision",
]
log_entries = []
for ticker in universe["Ticker"]:
    try:
        ticker_object = yf.Ticker(ticker)
        income_statement = ticker_object.income_stmt
        available_income_statement_fields = [field for field in income_statement_required_fields if field in income_statement.index]
        filtered_income_statement = income_statement.loc[available_income_statement_fields]
        if filtered_income_statement.empty:
            log_entries.append({
                "ticker": ticker, 
                "data_type": "income_statement",
                "status": "fail",
                "rows": 0,
                "start_date": None,
                "end_date": None,
                "missing_values": None,
                "missing_required_fields": None,
                "error": "empty data returned"
            }) 
        else:
            os.makedirs(f"../data/raw/fundamentals/{ticker}", exist_ok = True)
            cleaned_income_statement = filtered_income_statement.dropna(axis = 1, how = "all")
            cleaned_income_statement.to_csv(f"../data/raw/fundamentals/{ticker}/income_statement.csv")
            missing_values = cleaned_income_statement.isna().sum().sum()
            missing_fields = [field for field in income_statement_required_fields if field not in cleaned_income_statement.index]
            log_entries.append({
                "ticker": ticker, 
                "data_type": "income_statement",
                "status": "success",
                "rows": cleaned_income_statement.shape[1],
                "start_date": cleaned_income_statement.columns.min(),
                "end_date": cleaned_income_statement.columns.max(),
                "missing_values": missing_values,
                "missing_required_fields": missing_fields,
                "error": None
            })
    except Exception as e:
        log_entries.append({
            "ticker": ticker, 
            "data_type": "income_statement",
            "status": "fail",
            "rows": 0,
            "start_date": None,
            "end_date": None,
            "missing_values": None,
            "missing_required_fields": None,
            "error": str(e)
        })
collection_log = pd.DataFrame(log_entries)
collection_log.to_csv("../data/raw/collection_log.csv", index = False)